# 5-Minute High-Frequency Volatility Analysis

- Diurnal (time-of-day) variation
- Bipower Variation (BV)
- Jump-robust Truncated Variation (TV)
- Theoretical and Bootstrap 95% confidence intervals
- Annualised daily volatility plots

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

%matplotlib inline
plt.rcParams['figure.figsize'] = (13, 6)

## 2. Helper Functions — `YMDid` and `YMDid1`

Helper functions that convert packed integer date/time arrays into proper datetime objects.

- **`YMDid(DD)`** — two-column array `[YYYYMMDD, HHMM]` → `DatetimeIndex`  
- **`YMDid1(DD)`** — single-column array `[YYYYMMDD]` → `DatetimeIndex`

In [ ]:
def YMDid(DD):
    """Convert [YYYYMMDD, HHMM] array to pandas DatetimeIndex (replicates YMDid.m)."""
    dates = DD[:, 0].astype(int)
    times = DD[:, 1].astype(int)

    yyyy = dates // 10_000
    mm   = (dates - 10_000 * yyyy) // 100
    dd   = dates - yyyy * 10_000 - mm * 100

    hr = times // 100
    nn = times - hr * 100

    return pd.to_datetime(
        {"year": yyyy, "month": mm, "day": dd, "hour": hr, "minute": nn}
    )


def YMDid1(DD):
    """Convert [YYYYMMDD] array to pandas DatetimeIndex (replicates YMDid1.m)."""
    dates = DD[:, 0].astype(int)
    yyyy  = dates // 10_000
    mm    = (dates - 10_000 * yyyy) // 100
    dd    = dates - yyyy * 10_000 - mm * 100

    return pd.to_datetime({"year": yyyy, "month": mm, "day": dd})

## 3. Load Data

In [ ]:
mat = loadmat("IBM_5min.mat")
raw = mat["IBM5min"].astype(float)   # shape: (rows, cols)

n = 77                                # intraday return observations per day
T = raw.shape[0] // (n + 1)
print(f"Number of trading days T = {T}")
print(f"Intraday intervals     n = {n}")
print(f"Raw data shape           = {raw.shape}")

## 4. Extract Intraday Returns and Timestamps

Each trading day has `n+1` price observations → `n` log-returns.  
Returns are scaled to percentages: `r = 100 × Δlog(price)`.

In [ ]:
stk = np.zeros(n * T)       # intraday returns
DT  = np.zeros((n * T, 2))  # corresponding date-time stamps

for t in range(T):
    idx1 = np.arange(t * (n + 1), t * (n + 1) + (n + 1))     # n+1 prices
    idx2 = np.arange(t * n, t * n + n)                         # n returns
    idx3 = np.arange(t * (n + 1) + 1, t * (n + 1) + (n + 1)) # timestamps

    stk[idx2]   = 100 * np.diff(np.log(raw[idx1, 2]))
    DT[idx2, :] = raw[idx3, :2]

YMD_ret = YMDid(DT)           # full timestamp per return
YMD_TV  = YMD_ret[::n]        # one timestamp per day (first return of each day)

print(f"Returns vector length : {len(stk)}")
print(f"Date range            : {YMD_TV[0].date()} → {YMD_TV[-1].date()}")

## 5. Parameters

In [ ]:
alpha_J = 4.5        # jump-truncation threshold multiplier
delta_n = 1.0 / n    # sampling interval (fraction of trading day)

## 6. Diurnal (Time-of-Day) Variation

Estimates intraday seasonality in volatility using a bipower-type average across all days at each intraday interval `k`.  
The first interval copies its neighbour (Edge case Fix)

In [ ]:
mBVd = np.zeros(n)

for k in range(1, n):   # k = 1 … n-1 (0-indexed)
    IDa = np.arange(k, (T - 1) * n + k + 1, n)
    IDb = IDa - 1
    A   = stk[IDa]
    B   = stk[IDb]
    mBVd[k] = (np.pi / 2) * np.mean(np.abs(A) * np.abs(B))

mBVd[0] = mBVd[1]   # edge-case: copy neighbour

# Normalised diurnal factor
tods = (mBVd / mBVd.sum()) / delta_n

# Quick visualisation
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(np.arange(1, n + 1), tods)
ax.set_xlabel("Intraday interval")
ax.set_ylabel("Diurnal factor")
ax.set_title("IBM: Estimated diurnal volatility pattern")
plt.tight_layout()
plt.show()

## 7. Bipower Variation (BV)

Daily bipower variation is a jump-robust estimator of integrated variance:

$$\text{BV}_t = \frac{\pi}{2} \cdot \frac{n}{n-1} \sum_{i=2}^{n} |r_{t,i-1}| \cdot |r_{t,i}|$$

In [ ]:
BVs = np.zeros(T)

for t in range(T):
    idx1  = np.arange(t * n, t * n + n - 1)
    idx2  = idx1 + 1
    BVs[t] = (np.pi / 2) * n / (n - 1) * np.dot(np.abs(stk[idx1]), np.abs(stk[idx2]))

print(f"BV summary — mean: {BVs.mean():.4f}, min: {BVs.min():.4f}, max: {BVs.max():.4f}")

## 8. Jump-Robust Truncated Variation (TV)

Returns larger than `α_J × √(diurnal × BV) × Δn^0.49` are classified as jumps and excluded.  
TV is then the sum of squared truncated returns.  
TV_asym provides the asymptotic standard error (used in the theoretical CI).

In [ ]:
TV      = np.zeros(T)
TV_asym = np.zeros(T)

for t in range(T):
    id_    = np.arange(t * n, (t + 1) * n)
    thresh = alpha_J * np.sqrt(tods * BVs[t]) * delta_n ** 0.49
    keep_s = np.abs(stk[id_]) <= thresh
    tmp    = stk[id_] * keep_s

    TV[t]      = np.sum(tmp ** 2)
    TV_asym[t] = np.sqrt(np.sum(tmp ** 4) * 2 / 3)

print(f"TV summary — mean: {TV.mean():.4f}, min: {TV.min():.4f}, max: {TV.max():.4f}")

## 9. Bootstrap Confidence Intervals

A block bootstrap (block size `k_n = 11`) is applied within each day to construct empirical 95% CIs for TV.  
`Nsim = 100` simulations per day (increase for smoother CIs at the cost of runtime).

In [ ]:
k_n  = 11                  # block size
M    = n // k_n            # number of blocks
print(f"Block size k_n = {k_n}, number of blocks M = {M}")

TVq = np.zeros((T, 2))     # [lower 2.5%, upper 97.5%] quantiles
rng = np.random.default_rng(42)   # seeded for reproducibility

for t in range(T):
    id_    = np.arange(t * n, (t + 1) * n)
    thresh = alpha_J * np.sqrt(tods * BVs[t]) * delta_n ** 0.49
    keep_s = np.abs(stk[id_]) <= thresh
    rc     = keep_s * stk[id_]

    Nsim  = 100
    TVsim = np.zeros(Nsim)

    for j in range(Nsim):
        S = np.zeros(M)
        for K in range(M):
            blk_idx = np.arange(K * k_n, K * k_n + k_n)
            U       = rng.random(k_n)
            idS     = (np.ceil(U * k_n) - 1).astype(int)
            idd     = blk_idx[idS]
            S[K]    = np.sum(rc[idd] ** 2)
        TVsim[j] = np.sum(S)

    TVq[t, :] = np.quantile(TVsim, [0.025, 0.975])

print("Bootstrap complete.")

## 10. Plot — Annualised Volatility with Confidence Intervals

**Top panel**: Theoretical (asymptotic) 95% CI  
**Bottom panel**: Bootstrap 95% CI

In [ ]:
x_dates = YMD_TV.to_pydatetime()
ann_vol  = np.sqrt(TV * 252)
ci_half  = 1.96 * np.sqrt(252) / 2 / np.sqrt(TV) * TV_asym

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# ── Theoretical CI ──────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(x_dates, ann_vol,            '.k', markersize=4, label="TV (annualised)")
ax.plot(x_dates, ann_vol + ci_half,  'r-', linewidth=0.8, label="Upper 97.5%")
ax.plot(x_dates, ann_vol - ci_half,  'b-', linewidth=0.8, label="Lower 2.5%")
ax.set_ylim(0, 350)
ax.set_ylabel("Annualised volatility (%)")
ax.set_title("IBM: Annualised daily volatility — 95% Theoretical CI")
ax.legend(fontsize=8, loc="upper right")

# ── Bootstrap CI ─────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(x_dates, ann_vol,                       '.k', markersize=4, label="TV (annualised)")
ax.plot(x_dates, np.sqrt(TVq[:, 1] * 252),     'r-', linewidth=0.8, label="Upper 97.5%")
ax.plot(x_dates, np.sqrt(TVq[:, 0] * 252),     'b-', linewidth=0.8, label="Lower 2.5%")
ax.set_ylim(0, 350)
ax.set_ylabel("Annualised volatility (%)")
ax.set_xlabel("Date")
ax.set_title("IBM: Annualised daily volatility — 95% Bootstrap CI")
ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

## 11. Plot — Theoretical vs Bootstrap CI Width Comparison

Points on the 45° line indicate perfect agreement between the two CI methods.  
Deviations reveal days where the asymptotic approximation under- or over-estimates uncertainty.

In [ ]:
xx = 1.96 * np.sqrt(252) / 2 / np.sqrt(TV) * TV_asym * 2   # theoretical width
yy = np.sqrt(TVq[:, 1] * 252) - np.sqrt(TVq[:, 0] * 252)   # bootstrap width

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(xx, yy, '.', alpha=0.6, label="Days")
lim = max(xx.max(), yy.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k-', linewidth=1, label="45° line")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel("Width of theoretical CI")
ax.set_ylabel("Width of bootstrap CI")
ax.set_title("IBM: Theoretical vs Bootstrap CI width")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()